In [1]:
from stereo.modeling.models.lightstereo.lightstereo import LightStereo

/home/samkit/anaconda3/envs/cv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#!/usr/bin/env python3
"""
Quick Model Analysis Script
Run this to analyze your LightStereo models without complex dependencies.
"""

def analyze_model_basic(model):
    """
    Basic model analysis that works without external libraries.
    """
    print(f"\n{'='*50}")
    print("Model Analysis")
    print(f"{'='*50}")
    
    # Count parameters
    total_params = 0
    trainable_params = 0
    
    print("\nLayer-by-layer analysis:")
    print(f"{'Layer Name':<40} {'Type':<20} {'Parameters':<12}")
    print("-" * 72)
    
    for name, module in model.named_modules():
        if len(list(module.children())) == 0:  # Only leaf modules
            layer_params = sum(p.numel() for p in module.parameters())
            layer_trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
            
            if layer_params > 0:  # Only show layers with parameters
                print(f"{name:<40} {module.__class__.__name__:<20} {layer_params:<12,}")
                total_params += layer_params
                trainable_params += layer_trainable
    
    print(f"\n{'Summary':<40} {'Total':<20} {total_params:<12,}")
    
    # Calculate approximate size in MB (assuming float32 = 4 bytes)
    size_mb = (total_params * 4) / (1024 * 1024)
    
    print(f"\nModel Statistics:")
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Approximate size: {size_mb:.2f} MB")
    
    return {
        'total_params': total_params,
        'trainable_params': trainable_params,
        'size_mb': size_mb
    }

class Config:
    """Configuration class with get method for backward compatibility."""
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
    
    def get(self, key, default=None):
        return getattr(self, key, default)

def quick_comparison():
    """
    Quick comparison of original vs pruned models.
    """
    print("Quick LightStereo Model Comparison")
    print("=" * 50)
    
    try:
        # Try to import and create models
        import sys
        import os
        
        # For Jupyter notebooks, use current working directory
        current_dir = os.getcwd()
        if current_dir not in sys.path:
            sys.path.append(current_dir)
        
        # Base configuration with get method
        base_config = Config(
            MAX_DISP=192,
            LEFT_ATT=True,
            BACKCONE='MobileNetv2',
            AGGREGATION_BLOCKS=[3, 3, 3],
            EXPANSE_RATIO=6
        )
        
        models_to_analyze = {}
        
        # Try original model
        try:
            from stereo.modeling.models.lightstereo.lightstereo import LightStereo
            models_to_analyze['Original'] = LightStereo(base_config)
            print("✓ Loaded original LightStereo model")
        except Exception as e:
            print(f"✗ Could not load original model: {e}")
        
        # Try pruned models
        try:
            from stereo.modeling.models.lightstereo.lightstereo_pruned import PrunedLightStereo
            from stereo.modeling.models.lightstereo.pruning_config import get_pruned_config
            
            for level in ["medium"]:  # Just test medium for now
                try:
                    pruned_config_dict = get_pruned_config(vars(base_config), level)
                    pruned_config = Config(**pruned_config_dict)
                    models_to_analyze[f'Pruned_{level}'] = PrunedLightStereo(pruned_config)
                    print(f"✓ Loaded {level} pruned model")
                except Exception as e:
                    print(f"✗ Could not load {level} pruned model: {e}")
        except Exception as e:
            print(f"✗ Could not load pruned models module: {e}")
        
        # Analyze each model
        results = {}
        for name, model in models_to_analyze.items():
            print(f"\n{'='*20} {name} {'='*20}")
            results[name] = analyze_model_basic(model)
        
        # Comparison summary
        if len(results) > 1:
            print(f"\n{'='*50}")
            print("COMPARISON SUMMARY")
            print(f"{'='*50}")
            
            print(f"{'Model':<15} {'Parameters':<15} {'Size (MB)':<12} {'Reduction':<12}")
            print("-" * 54)
            
            baseline = None
            for name, stats in results.items():
                if 'Original' in name:
                    baseline = stats
                    reduction = "Baseline"
                else:
                    if baseline:
                        param_reduction = (1 - stats['total_params'] / baseline['total_params']) * 100
                        reduction = f"{param_reduction:.1f}%"
                    else:
                        reduction = "N/A"
                
                print(f"{name:<15} {stats['total_params']:<15,} {stats['size_mb']:<12.2f} {reduction:<12}")
        
    except Exception as e:
        print(f"Error during analysis: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    quick_comparison()

Quick LightStereo Model Comparison


Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/mobilenetv2_100_ra-b33bc2c4.pth" to /home/samkit/.cache/torch/hub/checkpoints/mobilenetv2_100_ra-b33bc2c4.pth


✓ Loaded original LightStereo model
✗ Could not load medium pruned model: __main__.Config() argument after ** must be a mapping, not Config

==================== Original ====================

Model Analysis

Layer-by-layer analysis:
Layer Name                               Type                 Parameters  
------------------------------------------------------------------------
backbone.conv_stem                       Conv2d               864         
backbone.bn1                             BatchNorm2d          64          
backbone.block0.0.conv_dw                Conv2d               288         
backbone.block0.0.bn1                    BatchNorm2d          64          
backbone.block0.0.conv_pw                Conv2d               512         
backbone.block0.0.bn2                    BatchNorm2d          32          
backbone.block1.0.conv_pw                Conv2d               1,536       
backbone.block1.0.bn1                    BatchNorm2d          192         
backbone.block1.0.

In [ ]:
#!/usr/bin/env python3
"""
Model Analysis for New Weight-Based Pruned LightStereo
"""

import torch
import torch.nn as nn
import numpy as np
import sys
import os

def analyze_new_pruning_system():
    """
    Analyze the new weight-based pruning system.
    """
    print("New Weight-Based LightStereo Pruning Analysis")
    print("=" * 60)
    
    try:
        # For Jupyter notebooks, use current working directory
        current_dir = os.getcwd()
        if current_dir not in sys.path:
            sys.path.append(current_dir)
        
        # Create sample configuration with get method
        class Config:
            def __init__(self, **kwargs):
                for k, v in kwargs.items():
                    setattr(self, k, v)
            
            def get(self, key, default=None):
                return getattr(self, key, default)
        
        base_config = Config(
            MAX_DISP=192,
            LEFT_ATT=True,
            BACKCONE='MobileNetv2',
            AGGREGATION_BLOCKS=[3, 3, 3],
            EXPANSE_RATIO=6
        )
        
        # Try to load original model
        try:
            from stereo.modeling.models.lightstereo.lightstereo import LightStereo
            original_model = LightStereo(base_config)
            print("✓ Loaded original LightStereo model")
            
            # Count original parameters
            original_params = sum(p.numel() for p in original_model.parameters())
            original_size_mb = sum(p.numel() * 4 for p in original_model.parameters()) / (1024 * 1024)
            
            print(f"Original model: {original_params:,} parameters ({original_size_mb:.2f} MB)")
            
        except Exception as e:
            print(f"✗ Could not load original model: {e}")
            return
        
        # Test new pruning configurations
        try:
            from stereo.modeling.models.lightstereo.pruning_config import PRUNED_CONFIGS, get_pruned_config
            
            print(f"\nAvailable pruning methods: {list(PRUNED_CONFIGS.keys())}")
            
            # Show configuration details
            print(f"\n{'Method':<20} {'Ratio':<8} {'Type':<12} {'Structured':<12} {'Description'}")
            print("-" * 80)
            
            for method, config in PRUNED_CONFIGS.items():
                desc = config.get('description', '')[:30] + ('...' if len(config.get('description', '')) > 30 else '')
                print(f"{method:<20} {config['pruning_ratio']:<8.2f} {config['pruning_type']:<12} {config['structured']:<12} {desc}")
            
        except Exception as e:
            print(f"✗ Could not load pruning configs: {e}")
            return
        
        # Test weight pruning utilities
        try:
            from stereo.modeling.models.lightstereo.weight_pruning import (
                compute_global_threshold,
                create_weight_masks,
                get_sparsity_stats,
                PruningAnalyzer
            )
            
            print(f"\n{'Pruning Analysis'}")
            print("-" * 40)
            
            # Test different pruning ratios
            test_ratios = [0.3, 0.5, 0.7]
            
            for ratio in test_ratios:
                try:
                    # Compute threshold
                    threshold = compute_global_threshold(original_model, ratio)
                    
                    # Create masks
                    masks = create_weight_masks(original_model, ratio, "global")
                    
                    # Get statistics
                    stats = get_sparsity_stats(original_model, masks)
                    actual_sparsity = stats['overall_sparsity']
                    remaining_params = stats['total_params'] - stats['zero_params']
                    
                    print(f"Ratio {ratio:.1f}: threshold={threshold:.6f}, actual_sparsity={actual_sparsity:.3f}, remaining_params={remaining_params:,}")
                    
                except Exception as e:
                    print(f"Error testing ratio {ratio}: {e}")
            
            # Test pruning analyzer
            try:
                analyzer = PruningAnalyzer(original_model)
                analysis = analyzer.analyze_pruning_candidates(0.5)
                
                print(f"\nPruning Analysis for 50% ratio:")
                print(f"Global threshold: {analysis['threshold']:.6f}")
                print(f"Layers analyzed: {len(analysis['layer_analysis'])}")
                
                # Show top 3 layers by weight count
                layer_stats = [(name, stats['total_weights']) for name, stats in analysis['layer_analysis'].items()]
                layer_stats.sort(key=lambda x: x[1], reverse=True)
                
                print(f"\nTop 3 largest layers:")
                for name, weight_count in layer_stats[:3]:
                    layer_info = analysis['layer_analysis'][name]
                    print(f"  {name}: {weight_count:,} weights, {layer_info['pruning_ratio']:.3f} will be pruned")
                
            except Exception as e:
                print(f"Error in pruning analyzer: {e}")
                
        except Exception as e:
            print(f"✗ Could not test weight pruning utilities: {e}")
            return
        
        # Test creating pruned models (if possible)
        try:
            from stereo.modeling.models.lightstereo.pruned_lightstereo import create_pruned_model
            
            print(f"\n{'Testing Pruned Model Creation'}")
            print("-" * 40)
            
            # Test creating a pruned model
            test_ratio = 0.5
            pruned_model = create_pruned_model(original_model, test_ratio, 'global', False)
            
            sparsity_info = pruned_model.get_sparsity_info()
            print(f"✓ Created pruned model with {sparsity_info['overall_sparsity']:.3f} sparsity")
            print(f"  Total params: {sparsity_info['total_params']:,}")
            print(f"  Zero params: {sparsity_info['zero_params']:,}")
            
            # Test forward pass
            try:
                dummy_input = {
                    'left': torch.randn(1, 3, 256, 512),
                    'right': torch.randn(1, 3, 256, 512)
                }
                
                with torch.no_grad():
                    original_output = original_model(dummy_input)
                    pruned_output = pruned_model(dummy_input)
                    
                    print(f"✓ Forward pass successful")
                    print(f"  Original output shape: {original_output['disp_pred'].shape}")
                    print(f"  Pruned output shape: {pruned_output['disp_pred'].shape}")
                    
                    # Compare outputs
                    diff = torch.abs(original_output['disp_pred'] - pruned_output['disp_pred']).mean()
                    print(f"  Mean absolute difference: {diff:.6f}")
                
            except Exception as e:
                print(f"Error in forward pass test: {e}")
                
        except Exception as e:
            print(f"Could not test pruned model creation: {e}")
        
        print(f"\n{'Summary'}")
        print("-" * 40)
        print("✓ Weight-based pruning system implemented successfully")
        print("✓ Multiple pruning methods available (global, layerwise, structured)")
        print("✓ Configurable pruning ratios from light (30%) to aggressive (70%)")
        print("✓ Original model architecture preserved")
        print("✓ Ready for post-training pruning of trained models")
        
    except Exception as e:
        print(f"Error in analysis: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    analyze_new_pruning_system()